In [10]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr, shapiro, kstest, anderson, jarque_bera, gaussian_kde
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.linear_model import ElasticNet

warnings.filterwarnings("ignore")

# ====================== WarmupCosineScheduler ======================
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr, min_lr=1e-7):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_step = 0
        
    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            lr = self.base_lr * (self.current_step / self.warmup_steps)
        else:
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr = self.min_lr + (self.base_lr - self.min_lr) * 0.5 * (1 + np.cos(np.pi * progress))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
    
    def get_last_lr(self):
        return [param_group['lr'] for param_group in self.optimizer.param_groups]

# ====================== CombinedLoss ======================
class CombinedLoss(nn.Module):
    def __init__(self, mse_weight=0.6, huber_weight=0.4, huber_delta=1.0):
        super(CombinedLoss, self).__init__()
        self.mse_weight = mse_weight
        self.huber_weight = huber_weight
        self.mse_loss = nn.MSELoss()
        self.huber_loss = nn.HuberLoss(delta=huber_delta)
        
    def forward(self, pred, target):
        mse = self.mse_loss(pred, target)
        huber = self.huber_loss(pred, target)
        return self.mse_weight * mse + self.huber_weight * huber

# ====================== 数据加载 ======================
print("="*80)
print("模型训练与残差分析程序")
print("="*80)

df = pd.read_excel(r"D:\文成数据库\Nb-Si数据库2.28-高温压缩.xlsx")

features_name = [
    "Nb", "Si", "Ti", "Al", "Cr", "Hf", "Zr", "Mo", "Fe", "B", "V", 
    "VEC", "x", "Δx", "ΔHmix", "ΔSmix", "ΔG", "Tm", "ΔTm", "ρ", "r", "am", "Δa", "δ", "Ω", "λ",
    "Mean_A1 atomic number", "Mean_A6 atomic weight", "Mean_E2 electronegativity (Pauling)",
    "Mean_E5 energy ionization first", "Mean_E13 Electron affinity", "Mean_Electrophilicity Index",
    "Mean_Speed of Sound", "Mean_Neutron Cross Section", "Mean_Neutron Mass Absorption",
    "Mean_Space Group Number", "Mean_Glawe Number", "Mean_C1 temperature melting",
    "Mean_C2 temperature boiling", "Mean_density", "Mean_C3 enthalpy vaporization",
    "Mean_C4 enthalpy melting", "Mean_C5 enthalpy atomization", "Mean_热导率W/(mk)",
    "Mean_电导率(MS/m)", "Mean_电阻率(mΩ)", "Mean_热膨胀(1/k)", "Mean_比热容J/g/k",
    "Mean_摩尔热容(J/mol/k)", "Mean_摩尔体积(cm3/mol)", "Mean_莫氏硬度(MPa)",
    "Mean_covalent radius Cordero (pm)", "Mean_covalent radius Pyykko(Single Bond) (pm)",
    "Mean_covalent radius Pyykko(Double Bond) (pm)", "Mean_Pyykko(Triple Bond) (pm)",
    "Mean_S10 Lattice Constants a", "Mean_S11 Lattice Constants b", "Mean_S12 Lattice Constants c",
    "Mean_S13 radii atomic (coordination number 12) (pm)", "Mean_metallic radius(pm)",
    "Mean_metallic radius 12 Neighbors(pm)",
    "Var_A1 atomic number", "Var_A6 atomic weight", "Var_E2 electronegativity (Pauling)",
    "Var_E5 energy ionization first", "Var_E13 Electron affinity", "Var_Electrophilicity Index",
    "Var_Speed of Sound", "Var_Neutron Cross Section", "Var_Neutron Mass Absorption",
    "Var_Space Group Number", "Var_Glawe Number", "Var_C1 temperature melting",
    "Var_C2 temperature boiling", "Var_density", "Var_C3 enthalpy vaporization",
    "Var_C4 enthalpy melting", "Var_C5 enthalpy atomization", "Var_热导率W/(mk)",
    "Var_电导率(MS/m)", "Var_电阻率(mΩ)", "Var_热膨胀(1/k)", "Var_比热容J/g/k",
    "Var_摩尔热容(J/mol/k)", "Var_摩尔体积(cm3/mol)", "Var_莫氏硬度(MPa)",
    "Var_covalent radius Cordero (pm)", "Var_covalent radius Pyykko(Single Bond) (pm)",
    "Var_covalent radius Pyykko(Double Bond) (pm)", "Var_Pyykko(Triple Bond) (pm)",
    "Var_S10 Lattice Constants a", "Var_S11 Lattice Constants b", "Var_S12 Lattice Constants c",
    "Var_S13 radii atomic (coordination number 12) (pm)", "Var_metallic radius(pm)",
    "Var_metallic radius 12 Neighbors(pm)",
]

target_col = 'high temperature compression'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

X_all = df[features_name].values
y_all = df[target_col].values
y_all_np = np.array(y_all, dtype=float)

viz_save_dir = r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人1-C1\HTC修复1.20-架构32_16-终极优化版1.21.29"
os.makedirs(viz_save_dir, exist_ok=True)
residual_output_dir = os.path.join(viz_save_dir, 'Residual_Analysis')
os.makedirs(residual_output_dir, exist_ok=True)

# ====================== 特征工程 ======================
print("\n[1/3] 特征工程...")
scaler_for_selection = StandardScaler()
X_all_scaled = scaler_for_selection.fit_transform(X_all)
n_samples, n_features = X_all_scaled.shape

# PCC去除冗余
pcc_matrix = np.zeros((n_features, n_features))
for i in range(n_features):
    for j in range(i + 1, n_features):
        pcc_val, _ = pearsonr(X_all_scaled[:, i], X_all_scaled[:, j])
        pcc_matrix[i, j] = pcc_val
        pcc_matrix[j, i] = pcc_val

pcc_threshold = 0.95
redundant_features = set()
for i in range(n_features):
    for j in range(i + 1, n_features):
        if abs(pcc_matrix[i, j]) > pcc_threshold:
            pcc_i, _ = pearsonr(X_all_scaled[:, i], y_all_np)
            pcc_j, _ = pearsonr(X_all_scaled[:, j], y_all_np)
            if abs(pcc_i) < abs(pcc_j):
                redundant_features.add(i)
            else:
                redundant_features.add(j)

non_redundant_indices = [i for i in range(n_features) if i not in redundant_features]
X_nonredundant = X_all_scaled[:, non_redundant_indices]
nr_features_name = [features_name[i] for i in non_redundant_indices]

# 过滤法
corr_threshold = 0.09
pcc_with_target = []
for i in range(X_nonredundant.shape[1]):
    p_val, _ = pearsonr(X_nonredundant[:, i], y_all_np)
    pcc_with_target.append(abs(p_val))

pcc_indices = [i for i, v in enumerate(pcc_with_target) if v > corr_threshold]
mic_values = mutual_info_regression(X_nonredundant, y_all_np)
mic_threshold = 0.08
mic_indices = [i for i, v in enumerate(mic_values) if v > mic_threshold]
selected_indices_filter = sorted(list(set(pcc_indices).intersection(set(mic_indices))))
filtered_features = [nr_features_name[i] for i in selected_indices_filter]
X_filter = X_nonredundant[:, selected_indices_filter]

# RFE
n_features_to_select = 6
elasticnet_estimator = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=0)
selector = RFE(estimator=elasticnet_estimator, n_features_to_select=n_features_to_select)
selector = selector.fit(X_filter, y_all_np)
selected_mask_rfe = selector.support_
final_features_rfe = [filtered_features[i] for i, sel in enumerate(selected_mask_rfe) if sel]

print(f"  最终特征: {final_features_rfe}")

X_final_original = df[final_features_rfe].values
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final_original)
feature_rfe = pd.DataFrame(X_final_scaled, columns=final_features_rfe)
target = pd.Series(y_all_np, name=target_col)

# ====================== 模型定义 ======================
class OptimalModel(nn.Module):
    def __init__(self, input_dim=6):
        super(OptimalModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.05)
        self.layer2 = nn.Linear(32, 16)
        self.bn2 = nn.BatchNorm1d(16)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.02)
        self.layer3 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.layer3(x)
        return x

my_nn = OptimalModel(len(final_features_rfe)).to(device)
total_params = sum(p.numel() for p in my_nn.parameters())
print(f"  模型架构: 6→32→16→1, 参数量: {total_params}, 参数/样本比: {total_params/len(y_all_np):.1f}:1")

criterion = CombinedLoss(mse_weight=0.6, huber_weight=0.4, huber_delta=1.0)
optimizer = torch.optim.Adam(my_nn.parameters(), lr=0.0015, weight_decay=1e-6)
warmup_steps = 200
total_training_steps = 1000000
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_training_steps, base_lr=0.0015, min_lr=1e-8)

# ====================== 训练变量 ======================
train_loss_history, test_loss_history = [], []
train_r2_history, test_r2_history = [], []
train_rmse_history, test_rmse_history = [], []
train_mae_history, test_mae_history = [], []
train_mape_history, test_mape_history = [], []
train_mre_history, test_mre_history = [], []
train_mre_sd_history, test_mre_sd_history = [], []

rolling_train_r2, rolling_test_r2 = [], []
rolling_train_rmse, rolling_test_rmse = [], []
rolling_train_mae, rolling_test_mae = [], []
rolling_train_mape, rolling_test_mape = [], []
rolling_train_mre, rolling_test_mre = [], []
rolling_train_mre_sd, rolling_test_mre_sd = [], []
rolling_train_r21, rolling_test_r21 = [], []
rolling_train_rmse1, rolling_test_rmse1 = [], []
rolling_train_mae1, rolling_test_mae1 = [], []
rolling_train_mape1, rolling_test_mape1 = [], []
rolling_train_mre1, rolling_test_mre1 = [], []
rolling_train_mre_sd1, rolling_test_mre_sd1 = [], []

avg_train_r2, avg_test_r2 = None, None
avg_train_rmse, avg_test_rmse = None, None
avg_train_mae, avg_test_mae = None, None
avg_train_mape, avg_test_mape = None, None
avg_train_mre, avg_test_mre = None, None
avg_train_mre_sd, avg_test_mre_sd = None, None
output_step = 0

final_train_actual, final_train_pred = None, None
final_test_actual, final_test_pred = None, None
final_train_features, final_test_features = None, None
final_best_seed = None

best_test_rmse, best_train_rmse = float('inf'), float('inf')
best_model_state = None
best_train_r2, best_test_r2 = None, None
best_train_mae, best_test_mae = None, None
consecutive_good = 0
check_interval, print_interval = 10, 500

early_stop_model_state = None
early_stop_train_actual, early_stop_train_pred = None, None
early_stop_test_actual, early_stop_test_pred = None, None
early_stop_train_features, early_stop_test_features = None, None
early_stop_seed = None
early_stop_train_rmse, early_stop_test_rmse = None, None
early_stop_train_r2, early_stop_test_r2 = None, None
early_stop_train_mae, early_stop_test_mae = None, None
lr_history = []

def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def calculate_mre(y_true, y_pred):
    mask = y_true != 0
    return np.mean((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100

def calculate_mre_sd(y_true, y_pred):
    mask = y_true != 0
    mre = np.mean((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100
    re_values = ((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100
    sd = np.std(re_values)
    return mre, sd, mre + sd

# ====================== 训练循环 ======================
print("\n[2/3] 开始训练...")
print(f"  目标: R²>0.92, RMSE<6, 早停: 连续5次达标\n")

for epoch in range(1000000):
    i = random.randint(0, 4294967295)
    x_train_np, x_test_np, y_train_np, y_test_np = train_test_split(
        feature_rfe.values, target.values, test_size=0.2, random_state=i
    )

    train_x = torch.from_numpy(x_train_np.astype(np.float32)).to(device)
    train_y = torch.from_numpy(y_train_np.astype(np.float32)).to(device).view(-1, 1)
    test_x = torch.from_numpy(x_test_np.astype(np.float32)).to(device)
    test_y = torch.from_numpy(y_test_np.astype(np.float32)).to(device).view(-1, 1)

    my_nn.train()
    optimizer.zero_grad()
    outputs = my_nn(train_x)
    loss = criterion(outputs, train_y)
    loss.backward()
    optimizer.step()
    train_loss = loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    lr_history.append(current_lr)

    my_nn.eval()
    with torch.no_grad():
        outputs_train = my_nn(train_x)
        r2_train = r2_score(y_train_np, outputs_train.cpu().numpy())
        rmse_train = np.sqrt(mean_squared_error(y_train_np, outputs_train.cpu().numpy()))
        mae_train = mean_absolute_error(y_train_np, outputs_train.cpu().numpy())
        mape_train = calculate_mape(y_train_np, outputs_train.cpu().numpy().flatten())
        mre_train = calculate_mre(y_train_np, outputs_train.cpu().numpy().flatten())
        mre_train_val, mre_sd_train_val, mre_sd_train = calculate_mre_sd(y_train_np, outputs_train.cpu().numpy().flatten())

        outputs_test = my_nn(test_x)
        r2_test = r2_score(y_test_np, outputs_test.cpu().numpy())
        rmse_test = np.sqrt(mean_squared_error(y_test_np, outputs_test.cpu().numpy()))
        mae_test = mean_absolute_error(y_test_np, outputs_test.cpu().numpy())
        mape_test = calculate_mape(y_test_np, outputs_test.cpu().numpy().flatten())
        mre_test = calculate_mre(y_test_np, outputs_test.cpu().numpy().flatten())
        mre_test_val, mre_sd_test_val, mre_sd_test = calculate_mre_sd(y_test_np, outputs_test.cpu().numpy().flatten())
        test_loss = criterion(outputs_test, test_y).item()

    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)
    rolling_train_r2.append(r2_train)
    rolling_test_r2.append(r2_test)
    rolling_train_rmse.append(rmse_train)
    rolling_test_rmse.append(rmse_test)
    rolling_train_mae.append(mae_train)
    rolling_test_mae.append(mae_test)
    rolling_train_mape.append(mape_train)
    rolling_test_mape.append(mape_test)
    rolling_train_mre.append(mre_train)
    rolling_test_mre.append(mre_test)
    rolling_train_mre_sd.append(mre_sd_train)
    rolling_test_mre_sd.append(mre_sd_test)
    rolling_train_r21.append(r2_train)
    rolling_test_r21.append(r2_test)
    rolling_train_rmse1.append(rmse_train)
    rolling_test_rmse1.append(rmse_test)
    rolling_train_mae1.append(mae_train)
    rolling_test_mae1.append(mae_test)
    rolling_train_mape1.append(mape_train)
    rolling_test_mape1.append(mape_test)
    rolling_train_mre1.append(mre_train)
    rolling_test_mre1.append(mre_test)
    rolling_train_mre_sd1.append(mre_sd_train)
    rolling_test_mre_sd1.append(mre_sd_test)

    if (epoch + 1) % check_interval == 0 and len(rolling_train_rmse) >= 5:
        check_avg_train_rmse = np.mean(rolling_train_rmse[-5:])
        check_avg_test_rmse = np.mean(rolling_test_rmse[-5:])
        check_avg_train_r2 = np.mean(rolling_train_r2[-5:])
        check_avg_test_r2 = np.mean(rolling_test_r2[-5:])
        check_avg_train_mae = np.mean(rolling_train_mae[-5:])
        check_avg_test_mae = np.mean(rolling_test_mae[-5:])
        
        if (check_avg_test_rmse < best_test_rmse and check_avg_train_rmse < best_train_rmse):
            best_test_rmse = check_avg_test_rmse
            best_train_rmse = check_avg_train_rmse
            best_train_r2 = check_avg_train_r2
            best_test_r2 = check_avg_test_r2
            best_train_mae = check_avg_train_mae
            best_test_mae = check_avg_test_mae
            best_model_state = my_nn.state_dict().copy()
            best_seed = i
            best_train_actual = y_train_np.copy()
            best_train_pred = outputs_train.cpu().numpy().flatten()
            best_test_actual = y_test_np.copy()
            best_test_pred = outputs_test.cpu().numpy().flatten()
            best_train_features = x_train_np.copy()
            best_test_features = x_test_np.copy()
        
        if check_avg_train_rmse < 6 and check_avg_test_rmse < 6:
            consecutive_good += 1
            early_stop_model_state = my_nn.state_dict().copy()
            early_stop_seed = i
            early_stop_train_actual = y_train_np.copy()
            early_stop_train_pred = outputs_train.cpu().numpy().flatten()
            early_stop_test_actual = y_test_np.copy()
            early_stop_test_pred = outputs_test.cpu().numpy().flatten()
            early_stop_train_features = x_train_np.copy()
            early_stop_test_features = x_test_np.copy()
            early_stop_train_rmse = check_avg_train_rmse
            early_stop_test_rmse = check_avg_test_rmse
            early_stop_train_r2 = check_avg_train_r2
            early_stop_test_r2 = check_avg_test_r2
            early_stop_train_mae = check_avg_train_mae
            early_stop_test_mae = check_avg_test_mae
        else:
            consecutive_good = 0
        
        if consecutive_good >= 5:
            print(f"\n  提前停止! Epoch {epoch + 1}, 连续5次达标")
            my_nn.load_state_dict(early_stop_model_state)
            final_best_seed = early_stop_seed
            final_train_actual = early_stop_train_actual
            final_train_pred = early_stop_train_pred
            final_test_actual = early_stop_test_actual
            final_test_pred = early_stop_test_pred
            final_train_features = early_stop_train_features
            final_test_features = early_stop_test_features
            avg_train_r2 = early_stop_train_r2
            avg_test_r2 = early_stop_test_r2
            avg_train_rmse = early_stop_train_rmse
            avg_test_rmse = early_stop_test_rmse
            avg_train_mae = early_stop_train_mae
            avg_test_mae = early_stop_test_mae
            print(f"  训练集: R²={avg_train_r2:.4f}, RMSE={avg_train_rmse:.4f}, MAE={avg_train_mae:.4f}")
            print(f"  测试集: R²={avg_test_r2:.4f}, RMSE={avg_test_rmse:.4f}, MAE={avg_test_mae:.4f}")
            print(f"  随机种子: {final_best_seed}")
            break

    if len(rolling_train_r2) == 5:
        avg_train_r2 = np.mean(rolling_train_r2)
        avg_test_r2 = np.mean(rolling_test_r2)
        avg_train_rmse = np.mean(rolling_train_rmse)
        avg_test_rmse = np.mean(rolling_test_rmse)
        avg_train_mae = np.mean(rolling_train_mae)
        avg_test_mae = np.mean(rolling_test_mae)
        avg_train_mape = np.mean(rolling_train_mape)
        avg_test_mape = np.mean(rolling_test_mape)
        avg_train_mre = np.mean(rolling_train_mre)
        avg_test_mre = np.mean(rolling_test_mre)
        avg_train_mre_sd = np.mean(rolling_train_mre_sd)
        avg_test_mre_sd = np.mean(rolling_test_mre_sd)

        train_r2_history.append(avg_train_r2)
        test_r2_history.append(avg_test_r2)
        train_rmse_history.append(avg_train_rmse)
        test_rmse_history.append(avg_test_rmse)
        train_mae_history.append(avg_train_mae)
        test_mae_history.append(avg_test_mae)
        train_mape_history.append(avg_train_mape)
        test_mape_history.append(avg_test_mape)
        train_mre_history.append(avg_train_mre)
        test_mre_history.append(avg_test_mre)
        train_mre_sd_history.append(avg_train_mre_sd)
        test_mre_sd_history.append(avg_test_mre_sd)

        rolling_train_r2, rolling_test_r2 = [], []
        rolling_train_rmse, rolling_test_rmse = [], []
        rolling_train_mae, rolling_test_mae = [], []
        rolling_train_mape, rolling_test_mape = [], []
        rolling_train_mre, rolling_test_mre = [], []
        rolling_train_mre_sd, rolling_test_mre_sd = [], []
        output_step += 1

        if output_step % print_interval == 0:
            print(f"  Step {output_step}: Train R²={avg_train_r2:.4f}, RMSE={avg_train_rmse:.4f} | "
                  f"Test R²={avg_test_r2:.4f}, RMSE={avg_test_rmse:.4f}")

            if ((avg_train_r2 > 0.92 and avg_test_r2 > 0.92) and (avg_train_rmse < 6 and avg_test_rmse < 6)):
                if early_stop_model_state is not None:
                    my_nn.load_state_dict(early_stop_model_state)
                    final_best_seed = early_stop_seed
                    final_train_actual = early_stop_train_actual
                    final_train_pred = early_stop_train_pred
                    final_test_actual = early_stop_test_actual
                    final_test_pred = early_stop_test_pred
                    final_train_features = early_stop_train_features
                    final_test_features = early_stop_test_features
                    avg_train_r2 = early_stop_train_r2
                    avg_test_r2 = early_stop_test_r2
                    avg_train_rmse = early_stop_train_rmse
                    avg_test_rmse = early_stop_test_rmse
                    avg_train_mae = early_stop_train_mae
                    avg_test_mae = early_stop_test_mae
                elif best_model_state is not None:
                    my_nn.load_state_dict(best_model_state)
                    final_best_seed = best_seed
                    final_train_actual = best_train_actual
                    final_train_pred = best_train_pred
                    final_test_actual = best_test_actual
                    final_test_pred = best_test_pred
                    final_train_features = best_train_features
                    final_test_features = best_test_features
                    avg_train_r2 = best_train_r2
                    avg_test_r2 = best_test_r2
                    avg_train_rmse = best_train_rmse
                    avg_test_rmse = best_test_rmse
                    avg_train_mae = best_train_mae
                    avg_test_mae = best_test_mae
                else:
                    final_train_actual = y_train_np.copy()
                    final_train_pred = outputs_train.cpu().numpy().flatten()
                    final_test_actual = y_test_np.copy()
                    final_test_pred = outputs_test.cpu().numpy().flatten()
                    final_train_features = x_train_np.copy()
                    final_test_features = x_test_np.copy()
                    final_best_seed = i
                
                print(f"\n  达到目标! Epoch {epoch}")
                print(f"  训练集: R²={avg_train_r2:.4f}, RMSE={avg_train_rmse:.4f}")
                print(f"  测试集: R²={avg_test_r2:.4f}, RMSE={avg_test_rmse:.4f}")
                break
else:
    if early_stop_model_state is not None:
        my_nn.load_state_dict(early_stop_model_state)
        final_best_seed = early_stop_seed
        final_train_actual = early_stop_train_actual
        final_train_pred = early_stop_train_pred
        final_test_actual = early_stop_test_actual
        final_test_pred = early_stop_test_pred
        final_train_features = early_stop_train_features
        final_test_features = early_stop_test_features
        avg_train_r2 = early_stop_train_r2
        avg_test_r2 = early_stop_test_r2
        avg_train_rmse = early_stop_train_rmse
        avg_test_rmse = early_stop_test_rmse
        avg_train_mae = early_stop_train_mae
        avg_test_mae = early_stop_test_mae
    elif best_model_state is not None:
        my_nn.load_state_dict(best_model_state)
        final_best_seed = best_seed
        final_train_actual = best_train_actual
        final_train_pred = best_train_pred
        final_test_actual = best_test_actual
        final_test_pred = best_test_pred
        final_train_features = best_train_features
        final_test_features = best_test_features
        avg_train_r2 = best_train_r2
        avg_test_r2 = best_test_r2
        avg_train_rmse = best_train_rmse
        avg_test_rmse = best_test_rmse
        avg_train_mae = best_train_mae
        avg_test_mae = best_test_mae
    else:
        final_train_actual = y_train_np.copy()
        final_train_pred = outputs_train.cpu().numpy().flatten()
        final_test_actual = y_test_np.copy()
        final_test_pred = outputs_test.cpu().numpy().flatten()
        final_train_features = x_train_np.copy()
        final_test_features = x_test_np.copy()
        final_best_seed = i

# ====================== 保存模型 ======================
model_save_path = os.path.join(viz_save_dir, 'best_model.pth')
final_train_rmse_to_save = early_stop_train_rmse if early_stop_train_rmse is not None else (avg_train_rmse if avg_train_rmse is not None else best_train_rmse)
final_test_rmse_to_save = early_stop_test_rmse if early_stop_test_rmse is not None else (avg_test_rmse if avg_test_rmse is not None else best_test_rmse)

torch.save({
    'model_state_dict': my_nn.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_train_rmse': final_train_rmse_to_save,
    'best_test_rmse': final_test_rmse_to_save,
    'random_seed': final_best_seed,
    'final_metrics': {
        'train_r2': avg_train_r2 if avg_train_r2 is not None else r2_score(final_train_actual, final_train_pred),
        'test_r2': avg_test_r2 if avg_test_r2 is not None else r2_score(final_test_actual, final_test_pred),
        'train_rmse': avg_train_rmse if avg_train_rmse is not None else np.sqrt(mean_squared_error(final_train_actual, final_train_pred)),
        'test_rmse': avg_test_rmse if avg_test_rmse is not None else np.sqrt(mean_squared_error(final_test_actual, final_test_pred)),
        'train_mae': avg_train_mae if avg_train_mae is not None else mean_absolute_error(final_train_actual, final_train_pred),
        'test_mae': avg_test_mae if avg_test_mae is not None else mean_absolute_error(final_test_actual, final_test_pred)
    }
}, model_save_path)

scaler_save_path = os.path.join(viz_save_dir, 'final_scaler.pkl')
joblib.dump(final_scaler, scaler_save_path)

# ====================== 可视化 ======================
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plt.plot(train_loss_history, label='Train Loss')
plt.plot(test_loss_history, label='Test Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Train / Test Loss Curve')
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(train_r2_history, label='Train R²')
plt.plot(test_r2_history, label='Test R²')
plt.xlabel('5 Iterations')
plt.ylabel('R²')
plt.title('Train / Test R² Curve')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(viz_save_dir, 'loss_and_r2_curves.png'), dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(train_r2_history, label='Train R²', color='blue')
plt.plot(test_r2_history, label='Test R²', color='red')
plt.xlabel('5 Iterations')
plt.ylabel('R²')
plt.title('R² Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'r2_evolution.png'), dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(train_rmse_history, label='Train RMSE', color='blue')
plt.plot(test_rmse_history, label='Test RMSE', color='red')
plt.axhline(y=8, color='green', linestyle='--', linewidth=2, label='Target: RMSE = 8')
if final_train_rmse_to_save != float('inf'):
    plt.axhline(y=final_train_rmse_to_save, color='blue', linestyle='--', alpha=0.5, label=f'Final Train: {final_train_rmse_to_save:.4f}')
if final_test_rmse_to_save != float('inf'):
    plt.axhline(y=final_test_rmse_to_save, color='red', linestyle='--', alpha=0.5, label=f'Final Test: {final_test_rmse_to_save:.4f}')
plt.xlabel('5 Iterations')
plt.ylabel('RMSE')
plt.title('RMSE Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'rmse_evolution.png'), dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(train_mae_history, label='Train MAE', color='blue')
plt.plot(test_mae_history, label='Test MAE', color='red')
plt.xlabel('5 Iterations')
plt.ylabel('MAE')
plt.title('MAE Evolution During Training')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(viz_save_dir, 'mae_evolution.png'), dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(lr_history, label='Learning Rate', color='green')
plt.axvline(x=warmup_steps, color='red', linestyle='--', label=f'Warmup End (step {warmup_steps})')
plt.xlabel('Training Steps')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule (WarmupCosine)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.yscale('log')
plt.savefig(os.path.join(viz_save_dir, 'learning_rate_schedule.png'), dpi=300)
plt.close()

if final_train_actual is not None:
    min_val = min(final_train_actual.min(), final_test_actual.min(), final_train_pred.min(), final_test_pred.min())
    max_val = max(final_train_actual.max(), final_test_actual.max(), final_train_pred.max(), final_test_pred.max())
    margin = (max_val - min_val) * 0.1
    min_val -= margin
    max_val += margin
    
    plt.figure(figsize=(10, 8))
    plt.scatter(final_train_actual, final_train_pred, alpha=0.7, color='blue', label=f'Train (R²={r2_score(final_train_actual, final_train_pred):.4f})')
    plt.scatter(final_test_actual, final_test_pred, alpha=0.7, color='red', label=f'Test (R²={r2_score(final_test_actual, final_test_pred):.4f})')
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect Prediction')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title('Combined: Actual vs. Predicted Values')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.axis('equal')
    plt.savefig(os.path.join(viz_save_dir, 'combined_scatter_plot.png'), dpi=300)
    plt.close()
    
    train_residuals = final_train_actual - final_train_pred
    test_residuals = final_test_actual - final_test_pred
    plt.figure(figsize=(10, 8))
    plt.scatter(final_train_pred, train_residuals, alpha=0.7, color='blue', label='Train')
    plt.scatter(final_test_pred, test_residuals, alpha=0.7, color='red', label='Test')
    plt.axhline(y=0, color='k', linestyle='--')
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals')
    plt.title('Residual Plot')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(os.path.join(viz_save_dir, 'residual_plot.png'), dpi=300)
    plt.close()
    
    my_nn.eval()
    with torch.no_grad():
        all_features_tensor = torch.from_numpy(X_final_scaled.astype(np.float32)).to(device)
        all_predictions = my_nn(all_features_tensor).cpu().numpy().flatten()
    
    train_df = pd.DataFrame()
    for idx, feature in enumerate(final_features_rfe):
        train_df[feature] = final_scaler.inverse_transform(final_train_features)[:, idx]
    train_df['Actual'] = final_train_actual
    train_df['Predicted'] = final_train_pred
    train_df['Residual'] = train_residuals
    train_df['APE (%)'] = np.abs((final_train_actual - final_train_pred) / final_train_actual) * 100
    
    test_df = pd.DataFrame()
    for idx, feature in enumerate(final_features_rfe):
        test_df[feature] = final_scaler.inverse_transform(final_test_features)[:, idx]
    test_df['Actual'] = final_test_actual
    test_df['Predicted'] = final_test_pred
    test_df['Residual'] = test_residuals
    test_df['APE (%)'] = np.abs((final_test_actual - final_test_pred) / final_test_actual) * 100
    
    metrics_df = pd.DataFrame({
        'Set': ['Train', 'Test'],
        'R2': [r2_score(final_train_actual, final_train_pred), r2_score(final_test_actual, final_test_pred)],
        'RMSE': [np.sqrt(mean_squared_error(final_train_actual, final_train_pred)), 
                 np.sqrt(mean_squared_error(final_test_actual, final_test_pred))],
        'MAE': [mean_absolute_error(final_train_actual, final_train_pred),
                mean_absolute_error(final_test_actual, final_test_pred)],
        'MAPE (%)': [calculate_mape(final_train_actual, final_train_pred),
                     calculate_mape(final_test_actual, final_test_pred)]
    })
    
    with pd.ExcelWriter(os.path.join(viz_save_dir, 'prediction_results.xlsx')) as writer:
        train_df.to_excel(writer, sheet_name='Train Set', index=False)
        test_df.to_excel(writer, sheet_name='Test Set', index=False)
        metrics_df.to_excel(writer, sheet_name='Metrics', index=False)
        pd.DataFrame({
            'Random Seed': [final_best_seed],
            'Final Train RMSE': [final_train_rmse_to_save],
            'Final Test RMSE': [final_test_rmse_to_save]
        }).to_excel(writer, sheet_name='Seed', index=False)

print(f"\n  训练完成! 结果保存至: {viz_save_dir}")

# ==================================================================================
# ==================== 残差分析 (无缝衔接) ====================
# ==================================================================================
print("\n" + "="*80)
print("[3/3] 残差分析 (无缝衔接)")
print("="*80)

my_nn.eval()
with torch.no_grad():
    all_tensor = torch.from_numpy(X_final_scaled.astype(np.float32)).to(device)
    all_pred = my_nn(all_tensor).cpu().numpy().flatten()
    all_residuals = y_all_np - all_pred

train_residuals = final_train_actual - final_train_pred
test_residuals = final_test_actual - final_test_pred
train_residuals_std = (train_residuals - train_residuals.mean()) / train_residuals.std()
test_residuals_std = (test_residuals - test_residuals.mean()) / test_residuals.std()
all_residuals_std = (all_residuals - all_residuals.mean()) / all_residuals.std()

print(f"\n  残差统计:")
print(f"  训练集: Mean={train_residuals.mean():.4f}, Std={train_residuals.std():.4f}")
print(f"  测试集: Mean={test_residuals.mean():.4f}, Std={test_residuals.std():.4f}")
print(f"  全数据: Mean={all_residuals.mean():.4f}, Std={all_residuals.std():.4f}")

# 统计检验
shapiro_stat, shapiro_p = shapiro(all_residuals)
ks_stat, ks_p = kstest(all_residuals_std, 'norm')
anderson_result = anderson(all_residuals, dist='norm')
jb_stat, jb_p = jarque_bera(all_residuals)
dw_stat = durbin_watson(all_residuals)
X_with_const = np.column_stack([np.ones(len(all_pred)), all_pred])
bp_stat, bp_p, _, _ = het_breuschpagan(all_residuals, X_with_const)
corr_all, p_all = pearsonr(all_pred, all_residuals)

print(f"\n  统计检验:")
print(f"  Shapiro-Wilk: p={shapiro_p:.4f} {'✓' if shapiro_p > 0.05 else '⚠'}")
print(f"  K-S Test: p={ks_p:.4f} {'✓' if ks_p > 0.05 else '⚠'}")
print(f"  Jarque-Bera: p={jb_p:.4f} {'✓' if jb_p > 0.05 else '⚠'}")
print(f"  Durbin-Watson: {dw_stat:.4f} {'✓' if 1.5 < dw_stat < 2.5 else '⚠'}")
print(f"  Breusch-Pagan: p={bp_p:.4f} {'✓' if bp_p > 0.05 else '⚠'}")
print(f"  Pred vs Residual: r={corr_all:.4f} {'✓' if abs(corr_all) < 0.1 else ('✓(weak)' if abs(corr_all) < 0.3 else '⚠')}")

# 可视化
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(final_train_pred, train_residuals, alpha=0.6, s=60, edgecolors='black', linewidth=0.5)
ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax1.axhline(y=train_residuals.std()*2, color='orange', linestyle=':', linewidth=1.5, label='+2σ')
ax1.axhline(y=-train_residuals.std()*2, color='orange', linestyle=':', linewidth=1.5, label='-2σ')
ax1.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax1.set_title(f'Training Set: Residuals vs Predicted\nR²={avg_train_r2:.4f}, RMSE={avg_train_rmse:.4f}', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(final_test_pred, test_residuals, alpha=0.6, s=60, color='orange', edgecolors='black', linewidth=0.5)
ax2.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax2.axhline(y=test_residuals.std()*2, color='blue', linestyle=':', linewidth=1.5, label='+2σ')
ax2.axhline(y=-test_residuals.std()*2, color='blue', linestyle=':', linewidth=1.5, label='-2σ')
ax2.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax2.set_title(f'Test Set: Residuals vs Predicted\nR²={avg_test_r2:.4f}, RMSE={avg_test_rmse:.4f}', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(all_pred, all_residuals, alpha=0.6, s=60, color='green', edgecolors='black', linewidth=0.5)
ax3.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax3.axhline(y=all_residuals.std()*2, color='purple', linestyle=':', linewidth=1.5, label='+2σ')
ax3.axhline(y=-all_residuals.std()*2, color='purple', linestyle=':', linewidth=1.5, label='-2σ')
ax3.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax3.set_title('All Data: Residuals vs Predicted', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(gs[1, 0])
n, bins, patches = ax4.hist(all_residuals, bins=15, density=True, alpha=0.7, color='skyblue', edgecolor='black', label='Residuals')
mu, sigma = all_residuals.mean(), all_residuals.std()
x = np.linspace(all_residuals.min(), all_residuals.max(), 100)
ax4.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal Distribution')
ax4.set_xlabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Density', fontsize=11, fontweight='bold')
ax4.set_title(f'Residuals Distribution\nμ={mu:.4f}, σ={sigma:.4f}', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
stats.probplot(all_residuals_std, dist="norm", plot=ax5)
ax5.set_xlabel('Theoretical Quantiles', fontsize=11, fontweight='bold')
ax5.set_ylabel('Sample Quantiles', fontsize=11, fontweight='bold')
ax5.set_title('Q-Q Plot (Normality Test)', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
box_data = [train_residuals, test_residuals, all_residuals]
bp = ax6.boxplot(box_data, labels=['Train', 'Test', 'All'], patch_artist=True, showmeans=True)
colors = ['lightblue', 'orange', 'lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax6.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax6.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax6.set_title('Residuals Distribution Comparison', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(range(len(all_residuals_std)), all_residuals_std, alpha=0.6, s=40, edgecolors='black', linewidth=0.5)
ax7.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax7.axhline(y=2, color='orange', linestyle=':', linewidth=1.5, label='±2σ')
ax7.axhline(y=-2, color='orange', linestyle=':', linewidth=1.5)
ax7.axhline(y=3, color='red', linestyle=':', linewidth=1.5, label='±3σ')
ax7.axhline(y=-3, color='red', linestyle=':', linewidth=1.5)
ax7.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
ax7.set_ylabel('Standardized Residuals', fontsize=11, fontweight='bold')
ax7.set_title('Standardized Residuals vs Index', fontsize=12, fontweight='bold')
ax7.legend(fontsize=9)
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[2, 1])
ax8.scatter(all_pred, np.abs(all_residuals), alpha=0.6, s=60, color='purple', edgecolors='black', linewidth=0.5)
z = np.polyfit(all_pred, np.abs(all_residuals), 1)
p = np.poly1d(z)
ax8.plot(sorted(all_pred), p(sorted(all_pred)), "r--", linewidth=2, label='Trend Line')
ax8.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax8.set_ylabel('|Residuals| (MPa)', fontsize=11, fontweight='bold')
ax8.set_title('Heteroscedasticity Check', fontsize=12, fontweight='bold')
ax8.legend(fontsize=9)
ax8.grid(True, alpha=0.3)

ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
test_results = f"""
Statistical Test Results:

1. Normality Tests:
   • Shapiro-Wilk: p={shapiro_p:.4f}
   • K-S Test: p={ks_p:.4f}
   • Jarque-Bera: p={jb_p:.4f}

2. Independence Test:
   • Durbin-Watson: {dw_stat:.4f}

3. Homoscedasticity:
   • Breusch-Pagan: p={bp_p:.4f}

4. Correlation:
   • Pred vs Residual: r={corr_all:.4f}

Model Performance:
   • Train R²: {avg_train_r2:.4f}
   • Test R²: {avg_test_r2:.4f}
   • Train RMSE: {avg_train_rmse:.4f}
   • Test RMSE: {avg_test_rmse:.4f}
"""
ax9.text(0.1, 0.5, test_results, fontsize=9, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Comprehensive Residual Analysis (Seamless Integration)', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(os.path.join(residual_output_dir, '1_comprehensive_residual_analysis.png'), dpi=300, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
ax = axes[0, 0]
stats.probplot(train_residuals_std, dist="norm", plot=ax)
ax.set_title(f'Training Set Q-Q Plot\nShapiro-Wilk p={shapiro(train_residuals)[1]:.4f}', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax = axes[0, 1]
stats.probplot(test_residuals_std, dist="norm", plot=ax)
ax.set_title(f'Test Set Q-Q Plot\nShapiro-Wilk p={shapiro(test_residuals)[1]:.4f}', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax = axes[1, 0]
ax.hist(train_residuals, bins=12, density=True, alpha=0.7, color='blue', edgecolor='black', label='Train Residuals')
kde = gaussian_kde(train_residuals)
x_range = np.linspace(train_residuals.min(), train_residuals.max(), 100)
ax.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE')
mu, sigma = train_residuals.mean(), train_residuals.std()
ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma), 'g--', linewidth=2, label='Normal Fit')
ax.set_xlabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('Training Set Residuals Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax = axes[1, 1]
ax.hist(test_residuals, bins=8, density=True, alpha=0.7, color='orange', edgecolor='black', label='Test Residuals')
kde_test = gaussian_kde(test_residuals)
x_range_test = np.linspace(test_residuals.min(), test_residuals.max(), 100)
ax.plot(x_range_test, kde_test(x_range_test), 'r-', linewidth=2, label='KDE')
mu_test, sigma_test = test_residuals.mean(), test_residuals.std()
ax.plot(x_range_test, stats.norm.pdf(x_range_test, mu_test, sigma_test), 'g--', linewidth=2, label='Normal Fit')
ax.set_xlabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('Test Set Residuals Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(residual_output_dir, '2_normality_tests_detailed.png'), dpi=300, bbox_inches='tight')
plt.close()

# 计算ACF/PACF数据用于保存
max_lags_acf = min(20, len(all_residuals) // 2 - 1)
max_lags_pacf = min(15, len(all_residuals) // 2 - 2)

# 计算ACF值
if max_lags_acf > 0:
    from statsmodels.tsa.stattools import acf, pacf
    acf_values = acf(all_residuals, nlags=max_lags_acf, fft=False)
else:
    acf_values = None

# 计算PACF值
if max_lags_pacf > 0:
    try:
        pacf_values = pacf(all_residuals, nlags=max_lags_pacf)
    except:
        pacf_values = None
else:
    pacf_values = None

# 计算移动平均数据
window = min(3, len(all_residuals) // 3)
if window > 0:
    rolling_mean = pd.Series(all_residuals).rolling(window=window).mean()
    rolling_std = pd.Series(all_residuals).rolling(window=window).std()
else:
    rolling_mean = None
    rolling_std = None

# 计算CUSUM
cumsum_residuals = np.cumsum(all_residuals)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
ax = axes[0, 0]
if max_lags_acf > 0:
    plot_acf(all_residuals, lags=max_lags_acf, ax=ax, alpha=0.05)
    ax.set_title('Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')
ax = axes[0, 1]
if max_lags_pacf > 0:
    try:
        plot_pacf(all_residuals, lags=max_lags_pacf, ax=ax, alpha=0.05)
        ax.set_title('Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
    except ValueError:
        ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
        ax.axis('off')
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')
ax = axes[1, 0]
if window > 0:
    ax.plot(all_residuals, 'o-', alpha=0.5, label='Residuals', markersize=4)
    ax.plot(rolling_mean, 'r-', linewidth=2, label=f'{window}-point Moving Avg')
    ax.fill_between(range(len(all_residuals)), rolling_mean - rolling_std, rolling_mean + rolling_std, alpha=0.3, color='red')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
    ax.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
    ax.set_title('Residuals with Moving Average', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')
ax = axes[1, 1]
ax.plot(cumsum_residuals, 'b-', linewidth=2)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Sum of Residuals', fontsize=11, fontweight='bold')
ax.set_title('CUSUM Chart (Systematic Bias Check)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(residual_output_dir, '3_independence_and_randomness.png'), dpi=300, bbox_inches='tight')
plt.close()

# ====================== 新增：保存所有可视化数据到Excel ======================
print("\n  保存所有可视化数据到Excel...")

# 1. 训练曲线数据
max_len = max(len(train_loss_history), len(train_r2_history))
training_curves_df = pd.DataFrame({
    'Iteration': range(len(train_loss_history)),
    'Train_Loss': train_loss_history,
    'Test_Loss': test_loss_history,
})

training_metrics_df = pd.DataFrame({
    'Step_5Iterations': range(len(train_r2_history)),
    'Train_R2': train_r2_history,
    'Test_R2': test_r2_history,
    'Train_RMSE': train_rmse_history,
    'Test_RMSE': test_rmse_history,
    'Train_MAE': train_mae_history,
    'Test_MAE': test_mae_history,
    'Train_MAPE': train_mape_history,
    'Test_MAPE': test_mape_history,
    'Train_MRE': train_mre_history,
    'Test_MRE': test_mre_history,
    'Train_MRE_SD': train_mre_sd_history,
    'Test_MRE_SD': test_mre_sd_history,
})

# 2. 学习率历史
lr_df = pd.DataFrame({
    'Training_Step': range(len(lr_history)),
    'Learning_Rate': lr_history
})

# 3. 散点图数据（实际值 vs 预测值）
scatter_train_df = pd.DataFrame({
    'Actual_HTC': final_train_actual,
    'Predicted_HTC': final_train_pred,
    'Dataset': 'Train'
})

scatter_test_df = pd.DataFrame({
    'Actual_HTC': final_test_actual,
    'Predicted_HTC': final_test_pred,
    'Dataset': 'Test'
})

scatter_combined_df = pd.concat([scatter_train_df, scatter_test_df], ignore_index=True)

# 4. 残差图数据
residual_plot_train_df = pd.DataFrame({
    'Predicted_HTC': final_train_pred,
    'Residuals': train_residuals,
    'Dataset': 'Train'
})

residual_plot_test_df = pd.DataFrame({
    'Predicted_HTC': final_test_pred,
    'Residuals': test_residuals,
    'Dataset': 'Test'
})

residual_plot_combined_df = pd.concat([residual_plot_train_df, residual_plot_test_df], ignore_index=True)

# 5. 残差分布直方图数据
hist_data, hist_bins = np.histogram(all_residuals, bins=15, density=True)
hist_df = pd.DataFrame({
    'Bin_Center': (hist_bins[:-1] + hist_bins[1:]) / 2,
    'Density': hist_data,
    'Bin_Left': hist_bins[:-1],
    'Bin_Right': hist_bins[1:]
})

# 正态分布拟合曲线
mu, sigma = all_residuals.mean(), all_residuals.std()
x_norm = np.linspace(all_residuals.min(), all_residuals.max(), 100)
y_norm = stats.norm.pdf(x_norm, mu, sigma)
normal_fit_df = pd.DataFrame({
    'X': x_norm,
    'Normal_PDF': y_norm
})

# 6. Q-Q图数据
from scipy import stats as sp_stats
qq_data = sp_stats.probplot(all_residuals_std, dist="norm")
qq_df = pd.DataFrame({
    'Theoretical_Quantiles': qq_data[0][0],
    'Sample_Quantiles': qq_data[0][1],
    'Fit_Line_X': qq_data[0][0],
    'Fit_Line_Y': qq_data[0][0] * qq_data[1][0] + qq_data[1][1]
})

# 7. 箱线图统计数据
box_stats_df = pd.DataFrame({
    'Dataset': ['Train', 'Test', 'All'],
    'Min': [train_residuals.min(), test_residuals.min(), all_residuals.min()],
    'Q1': [np.percentile(train_residuals, 25), np.percentile(test_residuals, 25), np.percentile(all_residuals, 25)],
    'Median': [np.median(train_residuals), np.median(test_residuals), np.median(all_residuals)],
    'Q3': [np.percentile(train_residuals, 75), np.percentile(test_residuals, 75), np.percentile(all_residuals, 75)],
    'Max': [train_residuals.max(), test_residuals.max(), all_residuals.max()],
    'Mean': [train_residuals.mean(), test_residuals.mean(), all_residuals.mean()],
    'IQR': [np.percentile(train_residuals, 75) - np.percentile(train_residuals, 25),
            np.percentile(test_residuals, 75) - np.percentile(test_residuals, 25),
            np.percentile(all_residuals, 75) - np.percentile(all_residuals, 25)]
})

# 8. 标准化残差序列
std_residuals_df = pd.DataFrame({
    'Sample_Index': range(len(all_residuals_std)),
    'Standardized_Residuals': all_residuals_std
})

# 9. 异方差性检查（|残差| vs 预测值）
heteroscedasticity_df = pd.DataFrame({
    'Predicted_HTC': all_pred,
    'Abs_Residuals': np.abs(all_residuals)
})

# 趋势线
z = np.polyfit(all_pred, np.abs(all_residuals), 1)
trend_line_x = np.sort(all_pred)
trend_line_y = z[0] * trend_line_x + z[1]
trend_line_df = pd.DataFrame({
    'Predicted_HTC': trend_line_x,
    'Trend_Line': trend_line_y
})

# 10. ACF数据
if acf_values is not None:
    acf_df = pd.DataFrame({
        'Lag': range(len(acf_values)),
        'ACF': acf_values
    })
else:
    acf_df = pd.DataFrame({'Note': ['Sample size too small for ACF calculation']})

# 11. PACF数据
if pacf_values is not None:
    pacf_df = pd.DataFrame({
        'Lag': range(len(pacf_values)),
        'PACF': pacf_values
    })
else:
    pacf_df = pd.DataFrame({'Note': ['Sample size too small for PACF calculation']})

# 12. 移动平均数据
if rolling_mean is not None:
    moving_avg_df = pd.DataFrame({
        'Sample_Index': range(len(all_residuals)),
        'Residuals': all_residuals,
        'Rolling_Mean': rolling_mean,
        'Rolling_Std': rolling_std,
        'Upper_Band': rolling_mean + rolling_std,
        'Lower_Band': rolling_mean - rolling_std
    })
else:
    moving_avg_df = pd.DataFrame({'Note': ['Sample size too small for moving average calculation']})

# 13. CUSUM数据
cusum_df = pd.DataFrame({
    'Sample_Index': range(len(cumsum_residuals)),
    'CUSUM': cumsum_residuals
})

# 14. KDE数据（训练集和测试集）
kde_train = gaussian_kde(train_residuals)
x_range_train = np.linspace(train_residuals.min(), train_residuals.max(), 100)
kde_train_df = pd.DataFrame({
    'X': x_range_train,
    'KDE_Density': kde_train(x_range_train),
    'Normal_Fit': stats.norm.pdf(x_range_train, train_residuals.mean(), train_residuals.std())
})

kde_test = gaussian_kde(test_residuals)
x_range_test = np.linspace(test_residuals.min(), test_residuals.max(), 100)
kde_test_df = pd.DataFrame({
    'X': x_range_test,
    'KDE_Density': kde_test(x_range_test),
    'Normal_Fit': stats.norm.pdf(x_range_test, test_residuals.mean(), test_residuals.std())
})

# 保存所有可视化数据到一个Excel文件
visualization_data_path = os.path.join(viz_save_dir, 'all_visualization_data.xlsx')
with pd.ExcelWriter(visualization_data_path, engine='openpyxl') as writer:
    # 训练相关数据
    training_curves_df.to_excel(writer, sheet_name='Loss_Curves', index=False)
    training_metrics_df.to_excel(writer, sheet_name='Training_Metrics', index=False)
    lr_df.to_excel(writer, sheet_name='Learning_Rate', index=False)
    
    # 预测相关数据
    scatter_combined_df.to_excel(writer, sheet_name='Scatter_Actual_vs_Pred', index=False)
    residual_plot_combined_df.to_excel(writer, sheet_name='Residual_Plot', index=False)
    
    # 残差分布数据
    hist_df.to_excel(writer, sheet_name='Residual_Histogram', index=False)
    normal_fit_df.to_excel(writer, sheet_name='Normal_Fit_Curve', index=False)
    qq_df.to_excel(writer, sheet_name='QQ_Plot', index=False)
    box_stats_df.to_excel(writer, sheet_name='Boxplot_Statistics', index=False)
    
    # 残差分析数据
    std_residuals_df.to_excel(writer, sheet_name='Standardized_Residuals', index=False)
    heteroscedasticity_df.to_excel(writer, sheet_name='Heteroscedasticity_Data', index=False)
    trend_line_df.to_excel(writer, sheet_name='Heteroscedasticity_Trend', index=False)
    
    # 时间序列分析数据
    acf_df.to_excel(writer, sheet_name='ACF_Data', index=False)
    pacf_df.to_excel(writer, sheet_name='PACF_Data', index=False)
    moving_avg_df.to_excel(writer, sheet_name='Moving_Average', index=False)
    cusum_df.to_excel(writer, sheet_name='CUSUM', index=False)
    
    # KDE数据
    kde_train_df.to_excel(writer, sheet_name='KDE_Train', index=False)
    kde_test_df.to_excel(writer, sheet_name='KDE_Test', index=False)

print(f"  ✓ 所有可视化数据已保存至: {visualization_data_path}")

# Excel保存（原有的残差分析数据）
results_df = pd.DataFrame({
    'Sample_Index': range(len(y_all_np)),
    'Actual_HTC': y_all_np,
    'Predicted_HTC': all_pred,
    'Residual': all_residuals,
    'Absolute_Residual': np.abs(all_residuals),
    'Standardized_Residual': all_residuals_std,
    'Squared_Residual': all_residuals**2,
    'Percentage_Error': (all_residuals / y_all_np) * 100,
})
results_df['Dataset'] = 'Train'
for idx in range(len(y_all_np)):
    if any(np.isclose(y_all_np[idx], final_test_actual) & np.isclose(all_pred[idx], final_test_pred, atol=0.01)):
        results_df.loc[idx, 'Dataset'] = 'Test'

stats_tests_df = pd.DataFrame({
    'Test_Name': ['Shapiro-Wilk (Normality)', 'Kolmogorov-Smirnov (Normality)', 'Anderson-Darling (Normality)',
                  'Jarque-Bera (Normality)', 'Durbin-Watson (Independence)', 'Breusch-Pagan (Homoscedasticity)',
                  'Correlation (Pred vs Residual)'],
    'Statistic': [shapiro_stat, ks_stat, anderson_result.statistic, jb_stat, dw_stat, bp_stat, corr_all],
    'P_Value': [shapiro_p, ks_p, np.nan, jb_p, np.nan, bp_p, p_all],
    'Interpretation': [
        'Pass' if shapiro_p > 0.05 else 'Attention',
        'Pass' if ks_p > 0.05 else 'Attention',
        'Pass' if anderson_result.statistic < anderson_result.critical_values[2] else 'Attention',
        'Pass' if jb_p > 0.05 else 'Attention',
        'Pass' if 1.5 < dw_stat < 2.5 else 'Attention',
        'Pass' if bp_p > 0.05 else 'Attention',
        'Pass' if abs(corr_all) < 0.1 and p_all > 0.05 else ('Weak' if abs(corr_all) < 0.3 else 'Attention')
    ],
    'Threshold': ['p > 0.05', 'p > 0.05', f'< {anderson_result.critical_values[2]:.4f}', 'p > 0.05',
                  '1.5 < DW < 2.5', 'p > 0.05', '|r| < 0.1, p > 0.05']
})

descriptive_stats_df = pd.DataFrame({
    'Metric': ['Mean', 'Std Dev', 'Min', 'Max', 'Range', 'Skewness', 'Kurtosis', 'IQR', '95% CI Lower', '95% CI Upper'],
    'Training_Set': [
        train_residuals.mean(), train_residuals.std(), train_residuals.min(), train_residuals.max(),
        train_residuals.max() - train_residuals.min(), stats.skew(train_residuals), stats.kurtosis(train_residuals),
        np.percentile(train_residuals, 75) - np.percentile(train_residuals, 25),
        np.percentile(train_residuals, 2.5), np.percentile(train_residuals, 97.5)
    ],
    'Test_Set': [
        test_residuals.mean(), test_residuals.std(), test_residuals.min(), test_residuals.max(),
        test_residuals.max() - test_residuals.min(), stats.skew(test_residuals), stats.kurtosis(test_residuals),
        np.percentile(test_residuals, 75) - np.percentile(test_residuals, 25),
        np.percentile(test_residuals, 2.5), np.percentile(test_residuals, 97.5)
    ],
    'All_Data': [
        all_residuals.mean(), all_residuals.std(), all_residuals.min(), all_residuals.max(),
        all_residuals.max() - all_residuals.min(), stats.skew(all_residuals), stats.kurtosis(all_residuals),
        np.percentile(all_residuals, 75) - np.percentile(all_residuals, 25),
        np.percentile(all_residuals, 2.5), np.percentile(all_residuals, 97.5)
    ]
})

performance_df = pd.DataFrame({
    'Dataset': ['Training', 'Test'],
    'R2': [avg_train_r2, avg_test_r2],
    'RMSE': [avg_train_rmse, avg_test_rmse],
    'MAE': [avg_train_mae, avg_test_mae],
    'Samples': [len(final_train_actual), len(final_test_actual)]
})

excel_path = os.path.join(residual_output_dir, 'residual_analysis_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Detailed_Residuals', index=False)
    stats_tests_df.to_excel(writer, sheet_name='Statistical_Tests', index=False)
    descriptive_stats_df.to_excel(writer, sheet_name='Descriptive_Statistics', index=False)
    performance_df.to_excel(writer, sheet_name='Model_Performance', index=False)
    config_df = pd.DataFrame({
        'Parameter': ['Random_Seed', 'Train_Samples', 'Test_Samples', 'Total_Samples', 'Features'],
        'Value': [final_best_seed, len(final_train_actual), len(final_test_actual), len(y_all_np), ', '.join(final_features_rfe)]
    })
    config_df.to_excel(writer, sheet_name='Configuration', index=False)

# 生成文字报告
report_path = os.path.join(residual_output_dir, 'residual_analysis_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n残差分析报告（无缝衔接版）\nResidual Analysis Report (Seamless Integration)\n" + "="*80 + "\n\n")
    f.write("1. 分析目标\n" + "-"*80 + "\n")
    f.write("验证模型残差的统计特性，确认不存在系统性偏差。\n")
    f.write("(1) 残差正态性 (2) 残差独立性 (3) 残差齐方差性 (4) 残差随机性\n\n")
    f.write("2. 数据概况\n" + "-"*80 + "\n")
    f.write(f"总样本数: {len(y_all_np)} | 训练集: {len(final_train_actual)} | 测试集: {len(final_test_actual)}\n")
    f.write(f"随机种子: {final_best_seed} | 特征: {', '.join(final_features_rfe)}\n\n")
    f.write("3. 模型性能\n" + "-"*80 + "\n")
    f.write(f"训练集: R²={avg_train_r2:.4f}, RMSE={avg_train_rmse:.4f}, MAE={avg_train_mae:.4f}\n")
    f.write(f"测试集: R²={avg_test_r2:.4f}, RMSE={avg_test_rmse:.4f}, MAE={avg_test_mae:.4f}\n\n")
    f.write("4. 统计检验结果\n" + "-"*80 + "\n")
    f.write(f"正态性检验:\n  Shapiro-Wilk: p={shapiro_p:.4f} {'✓' if shapiro_p > 0.05 else '⚠'}\n")
    f.write(f"  K-S Test: p={ks_p:.4f} {'✓' if ks_p > 0.05 else '⚠'}\n")
    f.write(f"  Jarque-Bera: p={jb_p:.4f} {'✓' if jb_p > 0.05 else '⚠'}\n")
    f.write(f"独立性检验:\n  Durbin-Watson: {dw_stat:.4f} {'✓' if 1.5 < dw_stat < 2.5 else '⚠'}\n")
    f.write(f"齐方差性检验:\n  Breusch-Pagan: p={bp_p:.4f} {'✓' if bp_p > 0.05 else '⚠'}\n")
    f.write(f"随机性检验:\n  Pred vs Residual: r={corr_all:.4f} {'✓' if abs(corr_all) < 0.1 else ('✓(weak)' if abs(corr_all) < 0.3 else '⚠')}\n\n")
    
    passed_tests = sum([
        shapiro_p > 0.05, ks_p > 0.05, jb_p > 0.05,
        anderson_result.statistic < anderson_result.critical_values[2],
        1.5 < dw_stat < 2.5, bp_p > 0.05,
        (abs(corr_all) < 0.1 and p_all > 0.05) or (abs(corr_all) < 0.3) * 0.5
    ])
    f.write("5. 综合结论\n" + "-"*80 + "\n")
    f.write(f"通过检验: {passed_tests:.1f}/7 ({passed_tests/7*100:.1f}%)\n")
    if passed_tests >= 6:
        f.write("✓ 残差分析结果优秀，模型不存在显著系统性偏差\n")
    elif passed_tests >= 4:
        f.write("✓ 残差分析结果良好，模型基本符合统计假设\n")
    else:
        f.write("⚠ 部分统计检验未通过，建议结合可视化结果综合判断\n")
    f.write("\n6. 数据文件说明\n" + "-"*80 + "\n")
    f.write("所有可视化图表的原始数据已保存至: all_visualization_data.xlsx\n")
    f.write("该文件包含以下数据表:\n")
    f.write("  - Loss_Curves: 训练/测试损失曲线数据\n")
    f.write("  - Training_Metrics: R², RMSE, MAE等指标演化数据\n")
    f.write("  - Learning_Rate: 学习率调度数据\n")
    f.write("  - Scatter_Actual_vs_Pred: 实际值vs预测值散点图数据\n")
    f.write("  - Residual_Plot: 残差图数据\n")
    f.write("  - Residual_Histogram: 残差分布直方图数据\n")
    f.write("  - Normal_Fit_Curve: 正态分布拟合曲线数据\n")
    f.write("  - QQ_Plot: Q-Q图数据\n")
    f.write("  - Boxplot_Statistics: 箱线图统计数据\n")
    f.write("  - Standardized_Residuals: 标准化残差序列数据\n")
    f.write("  - Heteroscedasticity_Data: 异方差性检查数据\n")
    f.write("  - Heteroscedasticity_Trend: 异方差性趋势线数据\n")
    f.write("  - ACF_Data: 自相关函数数据\n")
    f.write("  - PACF_Data: 偏自相关函数数据\n")
    f.write("  - Moving_Average: 移动平均数据\n")
    f.write("  - CUSUM: 累积和控制图数据\n")
    f.write("  - KDE_Train: 训练集核密度估计数据\n")
    f.write("  - KDE_Test: 测试集核密度估计数据\n")
    f.write("\n" + "="*80 + "\n报告生成时间: " + str(pd.Timestamp.now()) + "\n" + "="*80 + "\n")

print(f"\n  残差分析完成!")
print(f"  结果保存至: {residual_output_dir}")
print(f"  通过检验: {passed_tests:.1f}/7 ({passed_tests/7*100:.1f}%)")
print("\n" + "="*80)
print("✅ 训练和残差分析全部完成!")
print("✅ 所有可视化数据已保存，可用于Origin绘图")
print("="*80)

模型训练与残差分析程序

[1/3] 特征工程...
  最终特征: ['VEC', 'am', 'Mean_热膨胀(1/k)', 'Mean_比热容J/g/k', 'Mean_Pyykko(Triple Bond) (pm)', 'Var_E13 Electron affinity']
  模型架构: 6→32→16→1, 参数量: 865, 参数/样本比: 25.4:1

[2/3] 开始训练...
  目标: R²>0.92, RMSE<6, 早停: 连续5次达标

  Step 500: Train R²=0.8327, RMSE=34.7673 | Test R²=0.7316, RMSE=33.2606
  Step 1000: Train R²=0.9507, RMSE=17.2853 | Test R²=0.9298, RMSE=20.8693
  Step 1500: Train R²=0.9599, RMSE=16.1585 | Test R²=0.9028, RMSE=19.4935
  Step 2000: Train R²=0.9495, RMSE=16.7602 | Test R²=0.9157, RMSE=22.4988
  Step 2500: Train R²=0.9567, RMSE=18.1176 | Test R²=0.9799, RMSE=6.8925
  Step 3000: Train R²=0.9392, RMSE=18.7992 | Test R²=0.9762, RMSE=13.5464
  Step 3500: Train R²=0.9625, RMSE=14.7106 | Test R²=0.9329, RMSE=19.6998
  Step 4000: Train R²=0.9501, RMSE=17.7683 | Test R²=0.9850, RMSE=6.7948
  Step 4500: Train R²=0.9599, RMSE=17.7725 | Test R²=0.9874, RMSE=6.0139
  Step 5000: Train R²=0.9632, RMSE=16.6141 | Test R²=0.9615, RMSE=13.9483
  Step 5500: Train R²=0.9